# Ejemplo de practica:

## 1. Arquitectura y Extracción de Datos (Data Pipeline)
Para alimentar un dashboard analytics en D3.js v7, es fundamental estructurar los datos en formato JSON estandarizado.

### Fuentes de Datos Seleccionadas
- Mercado Español (IBEX 35): ^IBEX (Índice), SAN.MC (Banco Santander), e ITX.MC (Inditex).

- Mercado Estadounidense (Call Options): Contratos de opciones Call sobre AAPL, NVDA y SPY (extraídos vía Polygon.io o YFinance con fechas de vencimiento y strike fijos).

### Script de Descarga y Preparación (data_pipeline.py)
Este script obtiene las series temporales, calcula métricas financieras clave (rendimientos diarios, volatilidad móvil de 20 días y rendimiento acumulado base 100) y las exporta en JSON.

In [ ]:
#!pip install yfinance

In [3]:
import yfinance as yf
import pandas as pd
import numpy as np
import json

# 1. Ajustar calculate_analytics para conservar la longitud original (sin dropna)
def calculate_analytics(df):
    returns = df.pct_change() # Mantiene la fila inicial como NaN
    base_100 = (df / df.iloc[0]) * 100
    rolling_vol = returns.rolling(window=20).std() * np.sqrt(252)
    return returns, base_100, rolling_vol

# 2. Alineación explícita de calendarios bursátiles
# Combinamos y eliminamos fechas no coincidentes o rellenamos vacíos (ffill)
df_combined = pd.concat([data_ibex, data_options], axis=1, keys=['ibex', 'options']).dropna()

ibex_clean = df_combined['ibex']
options_clean = df_combined['options']

# Recalcular analítica sobre los datos alineados
ret_ibex, norm_ibex, vol_ibex = calculate_analytics(ibex_clean)
ret_opt, norm_opt, vol_opt = calculate_analytics(options_clean)

# 3. Construcción del JSON garantizando coincidencia de índices
dashboard_data = []
dates = df_combined.index.strftime('%Y-%m-%d').tolist()

for i, date in enumerate(dates):
    entry = {
        "date": date,
        "ibex": {
            "raw": {k: None if pd.isna(ibex_clean[k].iloc[i]) else float(ibex_clean[k].iloc[i]) for k in ibex_clean.columns},
            "norm": {k: None if pd.isna(norm_ibex[k].iloc[i]) else float(norm_ibex[k].iloc[i]) for k in norm_ibex.columns},
            "volatility": {k: None if pd.isna(vol_ibex[k].iloc[i]) else float(vol_ibex[k].iloc[i]) for k in vol_ibex.columns}
        },
        "options": {
            "raw": {k: None if pd.isna(options_clean[k].iloc[i]) else float(options_clean[k].iloc[i]) for k in options_clean.columns},
            "norm": {k: None if pd.isna(norm_opt[k].iloc[i]) else float(norm_opt[k].iloc[i]) for k in norm_opt.columns},
            "volatility": {k: None if pd.isna(vol_opt[k].iloc[i]) else float(vol_opt[k].iloc[i]) for k in vol_opt.columns}
        }
    }
    dashboard_data.append(entry)

# Matriz de correlación cruzada
corr_matrix = pd.concat([ret_ibex, ret_opt], axis=1).corr().fillna(0).to_dict()

with open('financial_data.json', 'w') as f:
    json.dump({"series": dashboard_data, "correlation": corr_matrix}, f, indent=2)

NameError: name 'data_ibex' is not defined

## 2. Definición Formática de Métricas Financieras

### - Normalización Base 100:
Para comparar activos con escalas dispares (ej. IBEX a 11,000 pts vs. Acción a 4 €):
$$P_{\text{norm}, t} = \frac{P_t}{P_0} \times 100$$

### Volatilidad Móvil Anualizada ($20$ días):
$$\sigma_{\text{anual}} = \sqrt{\frac{1}{N-1} \sum_{i=1}^{N} (R_i - \bar{R})^2} \times \sqrt{252}$$
donde $R_i = \ln(P_i / P_{i-1})$.

## 3. Desarrollo e Integración del Dashboard en D3.js (v7)
A continuación se presenta el código completo del dashboard. Incluye:

1. **IBEX 35**: Serie temporal multilínea con opción de alternar entre Precios Nominales y Rendimiento Relativo (Base 100).
2. **US Call Options**: Serie temporal multilínea para la evolución del valor de las primas.
3. **Heatmap**: Matriz de correlación entre activos tradicionales y derivados.
4. **Volatilidad Móvil**: Comparativo del perfil de riesgo acumulado.
5. **Interactividad:** Tooltip multilínea síncrono, filtro temporal con control tipo Brush y leyendas dinámicas.


Ver codigo (index_e.html)

Ver codigo (dashboard.js)


## 4. Funcionalidades de Análisis Financiero en el Dashboard
### 1. Gestión de Escalas Dispares:

Las series del IBEX 35 permiten alternar entre precio nominal y rendimiento relativo base 100, lo que previene que la magnitud del índice IBEX (~10,000 pts) oculte la variación porcentual de acciones de menor precio facial como Santander o Inditex.

### 2. Monitoreo de Volatilidad y Cobertura:

El gráfico de Volatilidad Móvil (20D) permite contrastar la estabilidad de los activos de renta variable tradicionales frente al apalancamiento y decaimiento temporal (theta) característico de las opciones Call estadounidenses.

### 3. Análisis de Correlación Cruzada:

El Heatmap de Correlación detecta relaciones co-dependientes entre las tendencias del mercado europeo (IBEX) y el apetito por riesgo tecnológico o global en EE. UU. (opciones sobre AAPL, NVDA y SPY).

### 4. Interactividad Sincronizada:

El control inferior mediante d3.brushX filtra dinámicamente la ventana temporal activa en todos los paneles, facilitando análisis específicos de periodos de alta volatilidad o eventos macroeconómicos.